# SNCP-PPO Social Navigation — Colab Notebook

End-to-end notebook for training and evaluating an **LTC + PPO** crowd-aware navigation policy on Google Colab.

## What you'll do

1. **Setup** — clone repo, install deps, mount Drive (optional)
2. **Smoke test** — verify the environment + model + training loop
3. **Train** — vectorized (parallel-env) curriculum training with multi-scenario holdout
4. **Evaluate** — characterize the trained policy on randomized scenarios
5. **Visualize** — trajectory plots + GIFs across all scenarios
6. **Analyze** — read training CSV, plot learning curves

## Architecture recap

- **Policy**: SNCPPolicy (3 LTC cells: temporal/spatial/node + attention + actor-critic heads)
- **Observation** (robot-local): robot_node (7), spatial_edges (**H×6** = position + relative velocity + **goal-direction unit vector** per pedestrian), temporal_edges (2)
- **Algorithm**: PPO with clipped value loss, GAE with truncation bootstrap, BPTT over LTC subsequences
- **Environment**: randomized layout — robot + pedestrians spawn at random antipodal points on a circle every episode (true generalization, not a memorized scene)
- **Training**: **vectorized** — N parallel envs collect 2048 transitions/update (fixes data starvation); curriculum N=1→5 driven by total env steps
- **Best-checkpoint metric**: `min(success across holdout scenarios)` — rewards generalists

## Colab tips

- **Runtime → Change runtime type → A100 (Colab Pro+)** recommended (~3-4h for a 2M-step run). Free-tier T4 works but slower.
- **Mount Drive (Section 1.4)** so checkpoints/logs persist if the session disconnects mid-run.
- The vectorized path batches N envs through one policy forward pass, so the GPU is used far better than the old single-env path — but env stepping (Social-Force sim) is still CPU-side, so more vCPUs (Pro+) also helps.

## 1. Setup

### 1.1 Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 1.2 Clone repository

Replace `heimdilon` with your GitHub user if you forked the repo.

In [ ]:
import os
REPO_URL = 'https://github.com/heimdilon/sncp-ppo-crowdnav.git'
REPO_DIR = '/content/sncp-ppo-crowdnav'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned. Pulling latest...')
    !cd {REPO_DIR} && git pull --rebase

%cd {REPO_DIR}
!ls -la

### 1.3 Install dependencies

PyTorch comes pre-installed on Colab; we just add `ncps`, `gymnasium`, and confirm versions.

In [ ]:
!pip install -q -r requirements.txt

import torch
print(f'torch    {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'         device: {torch.cuda.get_device_name(0)}')

import gymnasium, ncps, numpy, matplotlib
print(f'gymnasium {gymnasium.__version__}')
print(f'ncps      {ncps.__version__}')
print(f'numpy     {numpy.__version__}')
print(f'matplotlib {matplotlib.__version__}')

### 1.4 (Optional) Mount Google Drive

If you want checkpoints/logs to persist across Colab sessions, mount Drive and we'll symlink `checkpoints/` and `logs/` into a Drive folder.

**Skip this cell if you're just running a quick experiment.**

In [ ]:
USE_DRIVE = False  # set True to persist runs across Colab sessions
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/sncp-ppo-crowdnav-runs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(f'{DRIVE_PROJECT_DIR}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_PROJECT_DIR}/logs', exist_ok=True)
    # Replace local dirs with symlinks to Drive
    for sub in ('checkpoints', 'logs'):
        local = f'{REPO_DIR}/{sub}'
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.isdir(local):
            # Backup existing local dir then symlink
            import shutil
            for f in os.listdir(local):
                src = f'{local}/{f}'
                dst = f'{DRIVE_PROJECT_DIR}/{sub}/{f}'
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
            shutil.rmtree(local)
        os.symlink(f'{DRIVE_PROJECT_DIR}/{sub}', local)
    print(f'Drive-backed dirs: {DRIVE_PROJECT_DIR}/{{checkpoints,logs}}')
else:
    print('Drive mount skipped (USE_DRIVE=False). Files will be lost when Colab session ends.')

## 2. Smoke tests

Three fast self-tests:
1. Environment reset/step + observation shapes
2. Model forward pass
3. 50-episode mini-training (verifies the full pipeline + new Path A changes)

In [ ]:
!python test_env.py

In [ ]:
!python test_model.py

In [ ]:
# 50-episode smoke training — verifies curriculum, holdout, value clipping,
# LR schedule, return normalization, KL early-stop, holdout best-checkpoint
# warmup/threshold/tie-break (#14), and per-update diagnostics line
# (ent / kl / std / rms). Should complete in ~5 minutes on a T4 GPU.
# With replay 0 (default), every log line prints `[  PHASE Nh]`, never `[R …]`.
#
# Uses subprocess.run with a list-of-args instead of `!python ... \` so the
# IPython shell never has a chance to mangle multi-line continuations (which
# previously caused: argument --holdout_scenarios: invalid choice: ' ').
import subprocess, sys
cmd = [
    sys.executable, '-u', '-m', 'sncp_ppo.train',
    '--episodes', '50',
    '--num_humans', '5',
    '--seed', '42',
    '--eval_freq', '25',
    '--holdout_episodes', '3',
    '--holdout_scenarios', 'easy', 'hard',
    '--update_freq', '5',
    '--log_freq', '10',
    '--curriculum_replay_ratio', '0.0',
    '--save_path', 'checkpoints/sncp_ppo_smoke.pt',
]
print('Running:', ' '.join(cmd))
print('=' * 80)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print(f'\nExited with code {p.returncode}')

## 3. Full training (v12 — vectorized + goal-direction obs)

**v12** runs **N parallel environments** (`--num_envs`) with a fixed-horizon rollout, collecting `num_envs × horizon` transitions per PPO update (default 16×128 = **2048**). It adds each pedestrian's **goal-direction unit vector** to the observation (spatial_edges 4→6) so the policy can anticipate pedestrian trajectories — targeting the prediction-gap collisions diagnosed in v11 (87% of hard collisions were a pedestrian moving into the robot). Curriculum (1→5 pedestrians) and holdout best-checkpoint selection are driven by **total env steps** (`--total_steps`).

**v12 is a new architecture generation** (spatial input 6-dim): v8/v11 checkpoints cannot be loaded by this code. The single-env path (`--num_envs 1`) still exists for reproducibility.

### Key arguments (v12)

| Argument | Meaning | v12 value |
|---|---|---|
| `--num_envs` | Parallel envs (1 = legacy single-env path) | **16** |
| `--horizon` | Steps per env per PPO update | **128** |
| `--total_steps` | Env-step budget: drives curriculum + run length | **2_000_000** |
| `--eval_freq_updates` | Holdout cadence (in PPO updates) | **20** |
| `--lr` / `--target_kl` | Base lr / KL early-stop | **5e-5** / **0.01** |
| `--holdout_episodes` | Episodes per holdout per scenario | 50 |

~3-4h on an A100 (Colab Pro+). Mount Drive (Section 1.4) so a disconnect doesn't lose the run. **Success = randomized-hard eval > v11's 28%.**

In [ ]:
# Customize before running — v12: VECTORIZED training (N parallel envs) with
# step-budgeted curriculum + holdout. v12 adds each pedestrian's GOAL-DIRECTION
# to the observation (spatial_edges 4->6) so the policy can anticipate where
# pedestrians are heading — targeting the diagnosed prediction-gap collisions.
# NOTE: v12 is a new architecture generation; v8/v11 checkpoints cannot be loaded.
#
# Curriculum (1->5 pedestrians, 10/25/50/75% phases) and holdout best-checkpoint
# selection are driven by TOTAL ENV STEPS (--total_steps), not episodes.
# Envs are recreated at each phase boundary (all parallel envs must share
# num_humans). Best checkpoint = max min(success across holdout scenarios).
NUM_ENVS = 16         # parallel envs; 8 if GPU/CPU memory is tight
HORIZON = 128         # steps per env per update -> NUM_ENVS*HORIZON transitions/update
TOTAL_STEPS = 2_000_000   # env-step budget; drives curriculum + run length (~3-4h on A100)
SEED = 42
LR = 5e-5             # lowered from 1e-4 to damp holdout oscillation (v7/v8)
TARGET_KL = 0.01      # tighter than 0.015 default -> steadier convergence
SAVE_PATH = 'checkpoints/sncp_ppo_v12.pt'

import subprocess, sys
cmd = [
    sys.executable, '-u', '-m', 'sncp_ppo.train',
    '--num_envs', str(NUM_ENVS),
    '--horizon', str(HORIZON),
    '--total_steps', str(TOTAL_STEPS),
    '--eval_freq_updates', '20',
    '--num_humans', '5',
    '--seed', str(SEED),
    '--lr', str(LR),
    '--lr_end_factor', '0.1',
    '--target_kl', str(TARGET_KL),
    '--holdout_scenarios', 'easy', 'hard',
    '--holdout_episodes', '50',
    '--save_path', SAVE_PATH,
]
print('Running:', ' '.join(cmd))
print('=' * 80)
# Stream output line by line so we see progress live
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print(f'\nExited with code {p.returncode}')

### Resuming a previous Colab session

If Colab disconnected mid-training: with `USE_DRIVE=True`, your latest periodic checkpoint (`sncp_ppo_v3_colab_ep<N>.pt`) is still in Drive. There's no built-in resume CLI — easiest path is to just rerun from scratch with a different `--seed` (each seed gives a fresh trajectory). For exact resume support, see the **Roadmap** at the end of this notebook.

## 4. Evaluation

Run 100 deterministic episodes per scenario to characterize the trained policy.

In [ ]:
CHECKPOINT = 'checkpoints/sncp_ppo_v12.pt'  # best generalist from the vectorized v12 run
EVAL_SEED = 100  # different from training seeds for fair eval
EVAL_EPISODES = 100

# NOTE: randomized env (randomize_layout=True) -> these numbers measure TRUE
# generalization (fresh robot+pedestrian layout every episode), not a memorized
# scene. At n=100 the 95% CI is ~+/-9 points, so treat <10-point gaps as noise.
# Compare hard vs the v11 baseline (28%) to judge whether goal-direction obs helped.
# 'extreme' is the random-spawn OOD floor (curriculum trains on circle spawns).
for scenario in ['easy', 'easy_plus', 'medium', 'hard', 'extreme']:
    print(f'\n{"=" * 80}\nScenario: {scenario}\n{"=" * 80}')
    n_humans = {'easy': 1, 'easy_plus': 2, 'medium': 3, 'hard': 5, 'extreme': 5}[scenario]
    !python test_eval.py \
        --checkpoint {CHECKPOINT} \
        --num_humans {n_humans} \
        --scenario {scenario} \
        --n_episodes {EVAL_EPISODES} \
        --seed {EVAL_SEED} 2>&1 | tail -10

### Compare with shipped v2 baseline (if checkpoints included in repo)

In [ ]:
import os

if os.path.exists('checkpoints/sncp_ppo_v2.pt'):
    print('Evaluating v2 baseline (pre-Path-A, has known catastrophic-forgetting issues)')
    for scenario, n in [('easy', 1), ('hard', 5)]:
        print(f'\n--- v2 on {scenario}/{n}h ---')
        !python test_eval.py --checkpoint checkpoints/sncp_ppo_v2.pt \
            --num_humans {n} --scenario {scenario} --n_episodes 50 --seed 100 2>&1 | tail -8
else:
    print('v2 checkpoint not in repo. Train and save one, or clone the full repo with checkpoints.')

## 5. Visualize trajectories

Generate trajectory plots (PNG) and animated GIFs to inspect what the policy is doing visually.

In [ ]:
# Single trajectory plot — finds first successful episode out of 20 tries and plots it
!python visualize_trajectory.py \
    --checkpoint {CHECKPOINT} \
    --output trajectory_plot.png \
    --num_humans 5 \
    --scenario hard \
    --seed 42

from IPython.display import Image, display
display(Image('trajectory_plot.png'))

In [ ]:
# Animated GIF for a single scenario
!python visualize_trajectory_gif.py --checkpoint {CHECKPOINT}

from IPython.display import Image, display
import glob
gifs = sorted(glob.glob('*.gif'))
if gifs:
    print(f'Generated: {gifs}')
    display(Image(gifs[-1]))

In [ ]:
# All scenarios as separate GIFs (easy/medium/hard/extreme)
!python visualize_all_scenarios_gif.py --checkpoint {CHECKPOINT}

from IPython.display import Image, display
for sc in ['easy', 'medium', 'hard', 'extreme']:
    path = f'{sc}_trajectory.gif'
    if os.path.exists(path):
        print(f'\n--- {sc} ---')
        display(Image(path))

## 6. Training curves analysis

Plot the learning trajectory with per-scenario holdout lines and the generalist `min(success)` dashed line that drove best-checkpoint selection.

In [ ]:
import glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if not csv_files:
    print('No training CSVs found. Run section 3 first.')
else:
    latest_csv = csv_files[-1]
    print(f'Plotting: {latest_csv}')
    !python plot_training.py --csv {latest_csv} --output training_curves_colab.png --window 50
    from IPython.display import Image, display
    display(Image('training_curves_colab.png'))

### Inspect CSV in pandas (optional)

In [ ]:
import pandas as pd, glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if csv_files:
    df = pd.read_csv(csv_files[-1])
    print(f'Rows: {len(df)}')
    print(f'Columns: {list(df.columns)}')
    print('\nPhase distribution:')
    print(df['scenario'].value_counts().sort_index())
    print('\nHoldout summary (per-scenario success rate at each eval):')
    holdout_cols = [c for c in df.columns if c.startswith('holdout_') and c.endswith('_success')]
    if holdout_cols:
        # Take rows where holdout changed (event points only)
        hdf = df[holdout_cols].drop_duplicates()
        hdf.index = df.loc[hdf.index, 'episode']
        print(hdf.tail(10))

## 7. Persist results

If you used Drive (Section 1.4), checkpoints + logs already there. Otherwise download key artifacts before the session ends.

In [ ]:
from google.colab import files
import glob

DOWNLOAD = False  # set True to trigger browser download dialogs
if DOWNLOAD:
    # Best checkpoint
    if os.path.exists(SAVE_PATH):
        files.download(SAVE_PATH)
    # Latest training CSV + plot
    for pattern in ['logs/training_*.csv', 'training_curves_colab.png']:
        for f in sorted(glob.glob(pattern))[-1:]:
            files.download(f)

## 8. Notes & roadmap

### What v6 ships vs v3 and why v5 was rolled back

The v3 run reached 81% success on 5-human hard and 100% on 3-human medium,
but only 6% on the 1-human eval — classic catastrophic forgetting of the
low-density regime after 1350 episodes on N=3..5. v5 tried to fix that with
**interleaved curriculum replay** (re-sampling earlier phases for ~20% of
PPO update windows). It did help easy/easy_plus retention (holdout-easy
briefly hit 100% during the MEDIUM phase) but it **destroyed HARD-phase
learning**: rolling success dropped from 85% to 10%, best generalist went
from 13.3% to 0%, and HARD holdout collapsed to 0% at the end of training.

**Two failure modes combined:**
1. **Sample-budget hijack** — HARD lost ~75 of its 375 training episodes to
   replay of easier phases. At 1500 total episodes (already one order of
   magnitude below CrowdNav-RL literature), HARD couldn't reach the learning
   threshold.
2. **Return-RMS contamination** — `RunningMeanStd` saw mixed-distribution
   returns (N=1 → mostly positive, N=4 → mostly negative) and the running
   std drifted every 5 episodes, scrambling the critic's target scale.

v6 keeps the replay code in `train.py` but defaults `--curriculum_replay_ratio`
to **0.0**. Users can opt in for experiments; 1500-ep runs should leave it off.

| | v3 | v5 (regressed) | v6 (default) |
|---|---|---|---|
| Standstill penalty | 0 | 0 | 0 |
| Goal reward | +50 | +50 | +50 |
| Approach coef | 5 | 5 | 5 |
| Orientation weight | 0.05 | 0.05 | 0.05 |
| Collision penalty | -25 | -25 | -25 |
| Return normalization | RMS | RMS | RMS |
| KL early-stop | 0.015 | 0.015 | 0.015 |
| **Curriculum replay** | **off** | **0.2 (broken)** | **off (opt-in)** |
| **HARD rolling success** | **45-85%** | **0-10%** | **45-85% (expected)** |

### Known limitations

- **N=1 / N=2 forgetting**: v6 reproduces v3, so the 6% easy eval comes back.
  Fix planned via short post-training fine-tune on low-density phases only
  (see roadmap).
- **Random-spawn (`extreme`) is OOD**: the curriculum only trains on
  circle-pattern spawns. Mix random into a future phase if you need it.
- **Observation lacks human velocity**: policy must infer it via LTC dynamics
  → slow. Adding `(vx, vy)` to `spatial_edges` is **Yol B**.
- No exact training resume — only periodic checkpoint snapshots.

### Going further

1. **Post-training fine-tune (planned)** — load v6 ckpt, train 200 ep on
   N=1 and N=2 only with `lr=1e-5` and `target_kl=0.005` so HARD performance
   is preserved while easy/easy_plus is rescued. Solves the forgetting issue
   without paying the v5 budget cost.
2. **Hyperparameter sweep** — try `--seed 1,2,3,4,5` to bound noise.
3. **Multi-N PPO buffer** — pad `spatial_edges` to `max_humans=5` and use a
   mask so all phases share one rollout buffer. Then replay no longer steals
   sample budget. ~1-2 days of refactor (`models.py`, `ppo.py`, `crowd_env.py`).
4. **Architecture ablation** — swap one or more LTC cells with GRU to isolate
   LTC's contribution (Yol B).
5. **Real-robot sim2real** — `waffle_ros/` has a ROS node skeleton; calibrate
   noise and dynamics gap.

### Repo

- Source: https://github.com/heimdilon/sncp-ppo-crowdnav
- Issues / PRs welcome.